# Lesson 5.2 — System Monitoring: Telemetry and the Info Dictionaries

Monitoring **collects** the health signals every layer already emits (M6 manipulability, M8 effort, tracking error) into one dashboard.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, Fruit, model_perception,
    understand, to_configuration, plan_reference, execute_reference, track,
    system_monitor, fk_xy, P2, T2)
def pick_layer(seed_world=1, seed_perc=7):
    w = GreenhouseWorld.demo_row(n=6, seed=seed_world)
    dets = model_perception(w, rng=np.random.default_rng(seed_perc))
    _, tgt = understand(dets, w)
    layer = plan_reference(w.q.copy(), to_configuration(tgt), rng=np.random.default_rng(0))
    return w, tgt, layer
checks = []
w, tgt, layer = pick_layer()
rec = execute_reference(layer, telemetry=True)
mon = system_monitor(rec)
print('dashboard:', {k: round(val,4) for k, val in mon.items()})
for key in ['peak_error','rms','final_error_max','min_manipulability',
            'min_sigma_min','peak_effort','saturation_fraction','samples']:
    checks.append(key in mon)            # all expected health signals present


dashboard: {'samples': 566, 'peak_error': 0.0001, 'rms': 0.0001, 'final_error_max': 0.0001, 'min_manipulability': 0.0861, 'min_sigma_min': 0.1227, 'peak_effort': 4.6116, 'saturation_fraction': 0.0}


### Healthy run: small error/effort, manipulability bounded away from zero

In [2]:
print('peak_error=%.4f  peak_effort=%.2f  min_manip=%.4f' %
      (mon['peak_error'], mon['peak_effort'], mon['min_manipulability']))
checks.append(mon['peak_error'] < 0.01)
checks.append(mon['min_manipulability'] > 0.02)   # comfortably non-singular


peak_error=0.0001  peak_effort=4.61  min_manip=0.0861


### A disturbed run: the dashboard reflects elevated error and effort

In [3]:
def kick(t, j): return 50.0 if (j == 0 and t > 0.3) else 0.0
mon_d = system_monitor(execute_reference(layer, disturbance=kick, telemetry=True))
print('disturbed -> peak_error=%.3f  peak_effort=%.2f  min_manip=%.4f' %
      (mon_d['peak_error'], mon_d['peak_effort'], mon_d['min_manipulability']))
checks.append(mon_d['peak_error'] > mon['peak_error'])     # error rose
checks.append(mon_d['peak_effort'] > mon['peak_effort'])   # effort rose (controller strains)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


disturbed -> peak_error=1.836  peak_effort=58.68  min_manip=0.0861


12/12 checks passed.
All checks passed.
